# 12-Hour-Ahead Significant Wave Height Forecasting

This notebook compares several approaches to predicting `hsig_m` 12 hours in advance for the Mooloolaba wave buoy. All shared logic lives in `src/forecast/`; this notebook is the experimental surface.

**Problem.** Given wave buoy observations at 30-minute cadence up to time *t*, predict `hsig_m` at time *t + 12h* (i.e. 24 steps ahead).

**Why 12 hours?** It's the surf-forecast sweet spot: long enough that persistence ("it'll be the same as now") is no longer a great predictor, short enough that atmospheric chaos doesn't dominate.

**Evaluation.** Chronological 80/20 split. Metrics: MAE, RMSE, Bias, and skill score vs. persistence. A positive skill score is the honest test — any model that can't beat persistence has no predictive value.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge

import forecast as fc

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Horizon: {fc.HORIZON_HOURS}h = {fc.HORIZON_STEPS} steps at {fc.SAMPLING_FREQ_MINUTES}-minute cadence")

## 1. Load and inspect

The cleaned CSV comes from `python -m wave_data`. Gaps in the buoy record appear as NaN rows on the regular 30-minute grid; we'll handle those next.

In [ ]:
df_raw = fc.load_data()
print(f"{len(df_raw):,} rows spanning {df_raw.index.min().date()} \u2192 {df_raw.index.max().date()}")
print("\nMissing values per column:")
print(df_raw.isna().sum())
df_raw.head(3)

### Gap handling

Lag/rolling features propagate NaN. Three reasonable options:

1. **Drop rows with any NaN feature** — clean, but sacrifices recent data if a single channel fails.
2. **Interpolate** — reasonable for smooth geophysical signals over short gaps.
3. **Leave NaN and let the evaluation harness mask rows** — what our `evaluate()` does by default.

Short gaps in hsig_m/hmax_m/tz_s/tp_s are well-behaved (smooth series), so we'll interpolate. SST has the most missing values (~10%) but changes slowly, so interpolation is safe there too.

In [ ]:
df = df_raw.interpolate(limit=48).bfill().ffill()
assert df.isna().sum().sum() == 0, "expected fully imputed frame"
print(f"After imputation: {len(df):,} rows, 0 NaNs")

### Visualise hsig_m and its 12h-ahead autocorrelation

How much signal is even available at this horizon?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["hsig_m"].plot(ax=axes[0], alpha=0.7, title="hsig_m (2015\u20132025)")
axes[0].set_ylabel("Hs (m)")
lags_h = np.arange(0, 73, 1)  # 0..72h
ac = [df["hsig_m"].autocorr(lag=int(h * 2)) for h in lags_h]
axes[1].plot(lags_h, ac, marker=".")
axes[1].axvline(fc.HORIZON_HOURS, color="red", linestyle="--", label=f"{fc.HORIZON_HOURS}h horizon")
axes[1].set(xlabel="Lag (hours)", ylabel="Autocorrelation", title="hsig_m autocorrelation")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Autocorrelation at {fc.HORIZON_HOURS}h lag: {df['hsig_m'].autocorr(lag=fc.HORIZON_STEPS):.3f}")

## 2. Build target and feature matrix

Target: `y.loc[t] = hsig_m.loc[t + 12h]`, produced by `make_target`.

Features: start from the raw channels, then add

- **sin/cos of wave direction** (circular variable — otherwise the 359\u00b0\u21921\u00b0 jump looks like a 358\u00b0 swing to any regressor)
- **time-of-day and day-of-year cyclical encodings**
- **lag features** at 30min, 1h, 2h, 4h, 12h, 24h — capture short-term dynamics and the diurnal cycle
- **rolling means and stds** over 3h, 12h, 24h — smooth state + volatility proxy

All rolling features are shifted by 1 step so they never peek at the current observation (enforced by a test in `test_forecast.py`).

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    X = fc.encode_circular(df)
    X = fc.add_time_features(X)
    X = fc.add_lag_features(
        X,
        columns=["hsig_m", "hmax_m", "tp_s", "tz_s", "sst_c"],
        lags=[1, 2, 4, 8, 24, 48],
    )
    X = fc.add_rolling_features(
        X,
        columns=["hsig_m", "tp_s"],
        windows=[6, 24, 48],
        stats=("mean", "std"),
    )
    return X

X_full = build_features(df)
y_full = fc.make_target(df)

print(f"X shape: {X_full.shape}  |  feature count: {X_full.shape[1]}")
print(f"y NaN count (should equal horizon steps = {fc.HORIZON_STEPS}): {y_full.isna().sum()}")

In [ ]:
X_train, X_test, y_train, y_test = fc.chronological_split(X_full, y_full, test_frac=0.2)
print(f"train: {len(X_train):,}  ({X_train.index.min().date()} \u2192 {X_train.index.max().date()})")
print(f"test:  {len(X_test):,}  ({X_test.index.min().date()} \u2192 {X_test.index.max().date()})")

## 3. Baselines

- **Persistence:** ŷ = hsig\_m(t). With autocorrelation >0.8 at 12h (plot above), this is a tough baseline to beat.
- **Seasonal naive (24h):** ŷ = hsig\_m(t − 12h). Uses the fact that conditions 24h ago at the *target* time are correlated with now.
- **Climatology (hour-of-day):** ŷ = training-set mean of hsig\_m at the forecast hour. The floor any real model must beat.

Skill score is computed relative to persistence (the dominant baseline).

In [ ]:
persistence = fc.evaluate(
    fc.PersistenceForecaster(), X_train, y_train, X_test, y_test, name="persistence"
)
baseline_preds = persistence.predictions  # reference for skill-score-vs-persistence

seasonal = fc.evaluate(
    fc.SeasonalNaiveForecaster(period_steps=48), X_train, y_train, X_test, y_test,
    name="seasonal-naive-24h", baseline_preds=baseline_preds,
)
climatology = fc.evaluate(
    fc.ClimatologyHourForecaster(), X_train, y_train, X_test, y_test,
    name="climatology-hod", baseline_preds=baseline_preds,
)

fc.compare([persistence, seasonal, climatology])

## 4. Linear models

With 47 features and ~150k training rows, plain OLS is well-posed but likely to overfit on collinear lag columns. Ridge is the sensible default.

In [ ]:
linreg = fc.evaluate(
    LinearRegression(), X_train, y_train, X_test, y_test,
    name="linear-regression", baseline_preds=baseline_preds,
)
ridge = fc.evaluate(
    Ridge(alpha=1.0), X_train, y_train, X_test, y_test,
    name="ridge", baseline_preds=baseline_preds,
)
fc.compare([persistence, linreg, ridge])

## 5. Tree ensembles

Random Forest and Gradient Boosting capture non-linearities and feature interactions that lag-based linear models can't. Modest tree/leaf sizes keep this runnable on a laptop — a serious pass would tune via `TimeSeriesSplit` cross-validation.

In [ ]:
rf = fc.evaluate(
    RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
    X_train, y_train, X_test, y_test,
    name="random-forest", baseline_preds=baseline_preds,
)
gb = fc.evaluate(
    GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42),
    X_train, y_train, X_test, y_test,
    name="gradient-boosting", baseline_preds=baseline_preds,
)
fc.compare([persistence, ridge, rf, gb])

## 6. Full comparison

In [ ]:
all_results = [persistence, seasonal, climatology, linreg, ridge, rf, gb]
summary = fc.compare(all_results)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
summary["RMSE"].plot(kind="barh", ax=ax, color="#4c72b0")
ax.set(title=f"Test-set RMSE \u2014 {fc.HORIZON_HOURS}h hsig_m forecast", xlabel="RMSE (m)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Error analysis on the best model

Beyond a single RMSE number, we want to know *where* the model fails. Two useful views:

- Residuals across the test window — are there regimes (storms, calms) where we systematically under/over-predict?
- Error binned by observed hsig_m — does the model degrade for extreme swells, the events surfers care about most?

In [ ]:
best = min(all_results, key=lambda r: r.metrics["RMSE"])
print(f"Best model: {best.name}  |  RMSE = {best.metrics['RMSE']:.4f} m")

pred_series = pd.Series(best.predictions, index=X_test.index, name="pred")
resid = (y_test - pred_series).dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)
resid.plot(ax=axes[0], alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set(title=f"Residuals \u2014 {best.name}", ylabel="y \u2212 \u0177 (m)")

bins = pd.cut(y_test.dropna(), bins=[0, 0.75, 1.25, 1.75, 2.5, 10.0])
grouped = resid.groupby(bins, observed=True).agg(["mean", "std", "count"])
grouped.plot(y="std", kind="bar", ax=axes[1], legend=False, color="#55a868")
axes[1].set(title="Residual std by observed hsig_m bin", xlabel="Observed hsig_m (m)", ylabel="std of errors (m)")
plt.tight_layout()
plt.show()

grouped

## 8. Next steps

Ideas that fit the existing infrastructure without notebook rewrites:

- **Hyperparameter search with walk-forward CV** — wrap `evaluate()` in a loop over `sklearn.model_selection.TimeSeriesSplit` folds.
- **Exogenous inputs** — merge BOM wind/pressure fields or GFS reanalysis; at 12h+ horizons atmospheric forcing starts to matter more than local persistence.
- **Direct vs. recursive multi-step** — build a multi-output target (e.g. t+1..t+24) and compare direct (one model per horizon) against recursive (one-step model iterated). See where error accumulation dominates.
- **Gradient-boosted libs** — LightGBM / XGBoost often win on this kind of tabular time-series problem; add to `requirements.txt` and slot them into the same `evaluate()` harness.
- **Sequence models** — a small LSTM/Temporal Convolution on the recent window; only worth it if it beats Ridge by a meaningful margin.